# Quy trình thực nghiệm Hybrid Retrieval trên Colab
Chạy các cell theo thứ tự. Các bước có nhãn **[NGƯỜI]** cần tải file về, chỉnh sửa, rồi tải lên lại đúng thư mục trên Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/huuhieu56/PPL.git || (cd PPL && git pull)
%cd /content/PPL
!pip install -q -r requirements.in
!apt-get -qq install -y openjdk-17-jre-headless  # chỉ cần cho tokenizer VnCoreNLP (X3)

In [ ]:
import os
from google.colab import userdata
os.environ['PPL_DATA_DIR'] = '/content/drive/MyDrive/ppl-data/data'
os.environ['PPL_RUNS_DIR'] = '/content/drive/MyDrive/ppl-data/runs'
os.environ['PPL_VNCORENLP_DIR'] = '/content/drive/MyDrive/ppl-data/vncorenlp'
os.environ['PYTHONIOENCODING'] = 'utf-8'
for key in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_MODEL'):
    os.environ[key] = userdata.get(key)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU')

## 1. Lập chỉ mục
Đặt tài liệu PDF/DOCX/PPTX vào `MyDrive/ppl-data/docs/`. Tùy chọn: `docs_metadata.csv` (cột `filename, course, source_type, doc_title`), thêm `--metadata` vào lệnh.
Thay `VERSION` ở lệnh thứ hai bằng `version_id` vừa in ra (chỉ cần khi chạy X3 với VnCoreNLP).

In [ ]:
!python -m src.cli index build --input /content/drive/MyDrive/ppl-data/docs --course CS101 --tokenizers whitespace,pyvi
!python -m src.cli index add-tokenizer --index $PPL_DATA_DIR/indexes/VERSION --tokenizer vncorenlp  # thay VERSION

## 2. Sinh câu hỏi nháp và rà soát
**[NGƯỜI]** Sau `review-export`: tải `data/benchmark/review.csv` về, điền cột `action` (keep/edit/drop), `new_text`, `new_category`.
Viết thêm `human_queries.csv` (cột `text, category`) khoảng 25–30% tổng số câu, không nhìn tài liệu khi viết. Tải cả hai lên `MyDrive/ppl-data/data/benchmark/`.

In [ ]:
!python -m src.cli bench generate --per-category 60
!python -m src.cli bench review-export

In [ ]:
!python -m src.cli bench review-import --human $PPL_DATA_DIR/benchmark/human_queries.csv

## 3. Pooling và dán nhãn
**[NGƯỜI]** Hai người dán nhãn độc lập `annotation_A.csv` và `annotation_B.csv`: cột `relevance` (0/1/2), cột `evidence_quote` (trích nguyên văn đoạn căn cứ khi relevance ≥ 1).
Nếu lệnh agreement đầu tiên báo còn bất đồng: điền cột `final` trong `disagreements.csv`, rồi chạy lệnh thứ hai.

In [ ]:
!python -m src.cli bench pool --depth 15 --annotators A,B

In [ ]:
!python -m src.cli bench agreement --annotations $PPL_DATA_DIR/benchmark/annotation_A.csv $PPL_DATA_DIR/benchmark/annotation_B.csv
!python -m src.cli bench agreement --annotations $PPL_DATA_DIR/benchmark/annotation_A.csv $PPL_DATA_DIR/benchmark/annotation_B.csv --resolved $PPL_DATA_DIR/benchmark/disagreements.csv

In [ ]:
!python -m src.cli bench split --dev 0.3 --seed 42
!python -m src.cli bench describe

## 4. Tinh chỉnh trên dev và đánh giá trên test
Chỉ chạy `eval run --split test` sau khi đã chốt `frozen_params.yaml`. Mọi thay đổi tham số sau lần chạy test đầu tiên đều bị ghi nhận là vi phạm khóa.

In [ ]:
!python -m src.cli eval run --config configs/experiment.yaml --split test --dry-run
!python -m src.cli eval tune --config configs/experiment.yaml
!python -m src.cli eval run --config configs/experiment.yaml --split test

In [ ]:
import glob
RUN = sorted(glob.glob(os.environ['PPL_RUNS_DIR'] + '/*-test-*'))[-1]
print(RUN)
!python -m src.cli eval compare --config configs/experiment.yaml --run {RUN}
!python -m src.cli eval errors --config configs/experiment.yaml --run {RUN}

## 5. Phân tích lỗi và báo cáo
**[NGƯỜI]** Tải `error_sample.csv` trong thư mục run về, điền cột `cause` (extraction, chunk_boundary, tokenization, vocabulary_mismatch, multi_hop, label_error, other), tải lên lại rồi chạy cell dưới.

In [ ]:
!python -m src.cli eval errors --config configs/experiment.yaml --run {RUN} --summarize {RUN}/error_sample.csv
!python -m src.cli eval report --config configs/experiment.yaml --run {RUN}
from IPython.display import Markdown, Image, display
for path in sorted(glob.glob(RUN + '/report/table_*.md'), key=lambda p: int(p.rsplit('_', 1)[1][:-3])):
    display(Markdown(open(path, encoding='utf-8').read()))
for path in sorted(glob.glob(RUN + '/report/figure_*.png')):
    display(Image(path))